# Holiday windows — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## Setup — caricamento e controlli

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'axes.titleweight':'semibold', 'axes.labelsize':10, 'font.size':10})
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.max_rows', 80)

candidates = [
    Path.cwd() / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent.parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd() / 'adidas_sales_features.csv',
    Path.cwd().parent / 'adidas_sales_features.csv',
]
file_input = next((p for p in candidates if p.is_file()), None)
if file_input is None:
    raise FileNotFoundError('Manca output/adidas_sales_features.csv. Esegui i notebook 01 e 02 del progetto, poi avvia questo notebook dalla root o dalla cartella notebooks/.')
df = pd.read_csv(file_input, parse_dates=['invoice_date'], dtype={'retailer_id': 'string'})
expected = {'retailer','invoice_date','region','state','city','product','price_per_unit',
            'units_sold','total_sales','operating_profit','operating_margin','sales_method',
            'year','month_number','month_name_it','year_month','weekday_number',
            'weekday_name_it','is_weekend','season_it','product_gender','product_category'}
assert expected.issubset(df.columns), f'Colonne mancanti: {expected-set(df.columns)}'
assert not df['invoice_date'].isna().any(), 'Date mancanti'
assert np.allclose(df['total_sales'], df['price_per_unit']*df['units_sold'],atol=.011), 'Ricavi non coerenti'
assert np.allclose(df['operating_profit'],df['total_sales']*df['operating_margin'],atol=.011), 'Profitto non coerente'

EURO_MONTH=['Gennaio','Febbraio','Marzo','Aprile','Maggio','Giugno','Luglio','Agosto','Settembre','Ottobre','Novembre','Dicembre']
DAYS=['Lunedì','Martedì','Mercoledì','Giovedì','Venerdì','Sabato','Domenica']
SEASONS=['Inverno','Primavera','Estate','Autunno']

def units(x): return f'{x:,.0f}'.replace(',', '.')
def usd(x): return f'${x:,.2f}' if abs(x)<1000 else f'${x:,.0f}'
def percent(x): return 'N/D' if pd.isna(x) else f'{x:,.2f}%'
def safe(a,b): return np.nan if b==0 or pd.isna(b) else a/b
def avg(x): return f'{x:,.2f}'
def note(message): display(Markdown('**Risultato:** '+str(message)))
def caution(message): display(Markdown('**Limite interpretativo:** '+str(message)))
def data_table(frame, rows=80):
    display(frame.head(rows).reset_index(drop=True).round(2))
    if len(frame)>rows:
        display(Markdown(f'_Visualizzate {rows} righe su {len(frame)}. La tabella completa rimane disponibile nella variabile Python._'))

def summary(dims, data=None):
    frame=df if data is None else data
    group=frame.groupby(dims,dropna=False,observed=True)
    result=group.agg(units=('units_sold','sum'),revenue=('total_sales','sum'),
                     profit=('operating_profit','sum'),rows=('units_sold','size'),
                     price_mean=('price_per_unit','mean'),days=('invoice_date','nunique')).reset_index()
    result['margin_pct']=np.where(result.revenue!=0,100*result.profit/result.revenue,np.nan)
    result['realized_price']=np.where(result.units!=0,result.revenue/result.units,np.nan)
    result['profit_unit']=np.where(result.units!=0,result.profit/result.units,np.nan)
    return result

def dominance(frame,dims,category,measure='units'):
    x=frame.sort_values(dims+[measure,category],ascending=[True]*len(dims)+[False,True])
    first=x.drop_duplicates(subset=dims,keep='first').copy()
    top=frame.groupby(dims)[measure].transform('max')
    tied=frame[frame[measure].eq(top)].groupby(dims,dropna=False)[category].nunique().reset_index(name='ex_aequo')
    return first.merge(tied,on=dims,how='left')

def bar(frame,label,value,title,top=12,ascending=False,ax=None):
    d=frame.sort_values(value,ascending=ascending).head(top).copy()
    if ax is None: fig,ax=plt.subplots(figsize=(11,max(4,0.40*len(d)+1.5)))
    else: fig=ax.figure
    d=d.iloc[::-1]
    ax.barh(d[label].astype(str),d[value]);ax.set_title(title);ax.set_xlabel(value.replace('_',' ').title());ax.grid(axis='x',alpha=.25)
    ax.tick_params(axis='y',labelsize=9)
    fig.tight_layout()
    return ax

def twobars(frame,label,metrics,titles,top=12):
    fig,axes=plt.subplots(1,2,figsize=(15,max(4,0.43*min(top,len(frame))+1)))
    for ax,m,t in zip(axes,metrics,titles):bar(frame,label,m,t,top=top,ax=ax)
    plt.show()

def stack(data,index,columns,measure,title,top=14,shares=True):
    g=data.pivot_table(index=index,columns=columns,values=measure,aggfunc='sum',fill_value=0)
    if g.empty:
        note('Non sono presenti dati per questo confronto.');return
    g=g.loc[g.sum(axis=1).sort_values(ascending=False).head(top).index]
    if shares: g=g.div(g.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    ax=g.plot(kind='bar',stacked=True,figsize=(max(10, min(16,len(g)*0.7+5)),5.5),width=.80)
    ax.set_title(title);ax.set_ylabel('Quota sul volume (%)' if shares else measure)
    ax.set_xlabel('');ax.legend(title=columns,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

def heat(data,index,columns,value,title,top=25,normalize=False):
    piv=data.pivot_table(index=index,columns=columns,values=value,aggfunc='sum',fill_value=0)
    if piv.empty: return
    piv=piv.loc[piv.sum(axis=1).sort_values(ascending=False).head(top).index]
    if normalize: piv=piv.div(piv.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    fig,ax=plt.subplots(figsize=(max(8,1.15*len(piv.columns)+4),max(4,.36*len(piv)+1.8)))
    im=ax.imshow(piv.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(np.arange(len(piv.columns)),piv.columns.astype(str),rotation=35,ha='right')
    ax.set_yticks(np.arange(len(piv.index)),piv.index.astype(str));ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.80,label='Quota (%)' if normalize else value)
    fig.tight_layout();plt.show()

def lineplot(frame,x,y,title,color_key=None):
    fig,ax=plt.subplots(figsize=(12,5))
    if color_key is None:
        t=frame.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3)
    else:
        for name,piece in frame.groupby(color_key,observed=True):
            t=piece.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3,label=str(name))
        ax.legend(title=color_key)
    ax.set_title(title);ax.set_ylabel(y);ax.tick_params(axis='x',rotation=70,labelsize=8)
    ax.grid(alpha=.25);fig.tight_layout();plt.show()

note(f'Caricate {units(len(df))} righe • periodo {df.invoice_date.min():%Y-%m-%d} — {df.invoice_date.max():%Y-%m-%d} • nessun CSV verrà esportato.')

## Calendario delle festività e baseline

In [ ]:
# Giorni festivi effettivi negli USA. Finestre di analisi: ±3 giorni;
# baseline: 2–3 settimane prima/dopo, stesso giorno della settimana,
# escludendo date in ogni finestra festiva. Le finestre vicine possono sovrapporsi.
holiday_dates={
  2020:{'Independence Day':'2020-07-04','Thanksgiving':'2020-11-26','Black Friday':'2020-11-27','Cyber Monday':'2020-11-30','Natale':'2020-12-25'},
  2021:{'Independence Day':'2021-07-04','Thanksgiving':'2021-11-25','Black Friday':'2021-11-26','Cyber Monday':'2021-11-29','Natale':'2021-12-25'},
}
events=[(year,event,pd.Timestamp(date)) for year,names in holiday_dates.items() for event,date in names.items()]
all_event_days=set()
for year,event,date in events:
    all_event_days.update(pd.date_range(date-pd.Timedelta(days=3),date+pd.Timedelta(days=3)))

def holiday_pieces():
    pieces=[]
    for year,name,date in events:
        w=df.loc[df.invoice_date.between(date-pd.Timedelta(days=3),date+pd.Timedelta(days=3))].copy()
        if not w.empty:
            w['event']=name;w['event_year']=year;w['window']='Festività';pieces.append(w)
    return pd.concat(pieces,ignore_index=True)

fest=holiday_pieces()

def holiday_compare():
    results=[]
    for year,name,date in events:
        event_df=df[df.invoice_date.between(date-pd.Timedelta(days=3),date+pd.Timedelta(days=3))]
        refs=[]
        for lag in [-21,-14,14,21]:
            ref_date=date+pd.Timedelta(days=lag)
            dates=pd.date_range(ref_date-pd.Timedelta(days=3),ref_date+pd.Timedelta(days=3))
            refs.extend(d for d in dates if d not in all_event_days and d.year==year)
        base_df=df[df.invoice_date.isin(refs)]
        event_days=event_df.invoice_date.nunique();base_days=base_df.invoice_date.nunique()
        e_unit=safe(event_df.units_sold.sum(),event_days);b_unit=safe(base_df.units_sold.sum(),base_days)
        e_rev=safe(event_df.total_sales.sum(),event_days);b_rev=safe(base_df.total_sales.sum(),base_days)
        results.append({'year':year,'event':name,'fest_days':event_days,'base_days':base_days,
           'fest_units_day':e_unit,'base_units_day':b_unit,
           'fest_revenue_day':e_rev,'base_revenue_day':b_rev,
           'fest_profit_day':safe(event_df.operating_profit.sum(),event_days),
           'base_profit_day':safe(base_df.operating_profit.sum(),base_days),
           'unit_index_pct':100*safe(e_unit-b_unit,b_unit) if b_unit else np.nan,
           'fest_states':event_df.state.nunique(),'base_states':base_df.state.nunique(),
           'fest_retailers':event_df.retailer.nunique(),'base_retailers':base_df.retailer.nunique()})
    return pd.DataFrame(results)

## How do holiday-window metrics compare with nearby dates?

In [ ]:
t=holiday_compare().sort_values(['year','event']);valid=t.dropna(subset=['unit_index_pct']);best=valid.loc[valid.unit_index_pct.idxmax()]
note(f"Indice massimo per unità giornaliere rispetto alla baseline: {best['event']} {int(best.year)} ({percent(best.unit_index_pct)}). Indice descrittivo, NON effetto causale della festa.")
fig,ax=plt.subplots(figsize=(12,6));d=t.copy();d['label']=d.year.astype(str)+' – '+d.event
ax.barh(d.label,d.unit_index_pct);ax.axvline(0,color='black',lw=.8);ax.set_title('Scostamento volume giornaliero da baseline omologa (%)');ax.set_xlabel('% vs giorni di riferimento');plt.tight_layout();plt.show()
data_table(t)
caution('Finestre ±3 giorni sovrapposte intorno a Thanksgiving, Black Friday e Cyber Monday; nessuna somma fra eventi.')

## Which products and targets dominate holiday windows?

In [ ]:
p=summary(['event','product'],fest);c=summary(['event','product_category'],fest);g=summary(['event','product_gender'],fest)
w=dominance(p,['event'],'product')
note('Prodotto con più unità in ciascuna finestra festiva (anni aggregati, copertura non omogenea). I dati possono essere ripetuti tra finestre sovrapposte.')
heat(p,'event','product','units','Unità di prodotto per evento, finestre ±3 giorni')
data_table(w[['event','product','units','ex_aequo']]);data_table(c);data_table(g);data_table(p)

## Which retailers and territories contribute around holidays?

In [ ]:
r=summary(['event','retailer'],fest);loc=summary(['event','region'],fest);state=summary(['event','region','state'],fest);city=summary(['event','region','state','city'],fest)
w=dominance(r,['event'],'retailer');wl=dominance(loc,['event'],'region')
note('Primati per unità nelle finestre festive. Gli incroci non sono un confronto a perimetro commerciale costante.')
heat(r,'event','retailer','units','Eventi × retailer: unità')
data_table(w[['event','retailer','units']]);data_table(wl[['event','region','units']]);data_table(state.sort_values(['event','units'],ascending=[True,False]),rows=320);data_table(city.sort_values(['event','units'],ascending=[True,False]),rows=330)

## What is the sales-method mix around holidays?

In [ ]:
t=summary(['event','sales_method'],fest);t['share_units_pct']=100*t.units/t.groupby('event').units.transform('sum');t['share_rev_pct']=100*t.revenue/t.groupby('event').revenue.transform('sum')
w=dominance(t,['event'],'sales_method')
note('Il canale prevalente è calcolato per unità; le quote economiche sono presentate separatamente e si riferiscono a ciascuna finestra.')
stack(t,'event','sales_method','units','Canali nelle festività: composizione unità (%)')
data_table(w[['event','sales_method','units','share_units_pct','share_rev_pct']]);data_table(t)

## Can holiday windows be fairly compared between years?

In [ ]:
t=holiday_compare();p=t.pivot(index='event',columns='year',values='fest_units_day').fillna(0)
comparable=t.pivot(index='event',columns='year',values='fest_states')
note('NON CONFERMATO un confronto omogeneo 2020–2021: cambiano gli stati/retailer disponibili nelle finestre. Il grafico mostra solo unità per data osservata.')
p.plot(kind='bar',figsize=(12,6));plt.title('Volume medio per giorno osservato nelle finestre festive');plt.ylabel('Unità / data osservata');plt.xticks(rotation=25,ha='right');plt.tight_layout();plt.show()
data_table(t[['year','event','fest_days','fest_units_day','fest_revenue_day','fest_states','fest_retailers','base_days','base_states']])